# Barcode Detection with zxing-cpp

This notebook tests `zxing-cpp` as a direct decoder baseline. It lightly enhances each scanned page, tries common barcode formats, converts decoded locations into boxes, and displays the result across all 21 pages.


In [ ]:
from pathlib import Path
import subprocess
import shutil
import math

import cv2
import numpy as np
import zxingcpp
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data' / 'pdfs').exists():
    ROOT = ROOT.parent

PDF_PATH = ROOT / 'data' / 'pdfs' / 'Quality Dossier.pdf'

PAGES = list(range(1, 22))
OUT_DIR = ROOT / 'output' / 'zxing-cpp-atom'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300
GRID_COLS = 2
CELL_WIDTH = 600

def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'codex-runtimes'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')



Configure barcode formats, prepare page variants, call `zxing-cpp`, and convert decoder positions into drawable boxes.


In [ ]:
DECODER_FORMATS = zxingcpp.barcode_formats_from_str('Code39,Code128,DataMatrix,QRCode')

def read_barcodes(image):
    return zxingcpp.read_barcodes(
        image,
        formats=DECODER_FORMATS,
        try_rotate=True,
        try_downscale=True,
        try_invert=True,
        binarizer=zxingcpp.Binarizer.LocalAverage,
    )

def crop_edges(img_cv, margin=10):
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 250, 255, cv2.THRESH_BINARY_INV)
    coords = cv2.findNonZero(thresh)
    if coords is None:
        return img_cv
    x, y, w, h = cv2.boundingRect(coords)
    h_img, w_img = img_cv.shape[:2]
    x = max(0, x - margin)
    y = max(0, y - margin)
    w = min(w_img - x, w + 2 * margin)
    h = min(h_img - y, h + 2 * margin)
    return img_cv[y:y+h, x:x+w]

def detect_page(img_path):
    img_cv = cv2.imread(str(img_path))
    if img_cv is None:
        raise FileNotFoundError(f'Image not found: {img_path}')

    cropped = crop_edges(img_cv)
    barcodes = read_barcodes(cropped)

    colors = [
        (0, 255, 0), (0, 255, 255), (255, 0, 0),
        (0, 165, 255), (255, 0, 255), (0, 255, 165), (0, 255, 255),
    ]

    for i, b in enumerate(barcodes):
        pos = b.position
        color = colors[i % len(colors)]

        cv2.rectangle(img_cv, (pos.top_left.x, pos.top_left.y),
                      (pos.bottom_right.x, pos.bottom_right.y), color, 2)
        cv2.putText(img_cv, f'{b.format}: {b.text}',
                    (pos.top_left.x, pos.top_left.y - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    result = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
    return Image.fromarray(result), len(barcodes)


def create_grid(images, cols=2, cell_width=600):
    if not images:
        return None
    resized = []
    for img in images:
        ratio = cell_width / img.width
        new_h = int(img.height * ratio)
        resized.append(img.resize((cell_width, new_h), Image.LANCZOS))
    n = len(resized)
    rows_count = math.ceil(n / cols)
    row_heights = []
    for r in range(rows_count):
        start = r * cols
        end = min(start + cols, n)
        row_heights.append(max(resized[i].height for i in range(start, end)))
    total_h = sum(row_heights)
    grid = Image.new('RGB', (cell_width * cols, total_h))
    y = 0
    for r in range(rows_count):
        start = r * cols
        end = min(start + cols, n)
        for idx in range(start, end):
            img = resized[idx]
            x = (idx % cols) * cell_width
            y_offset = (row_heights[r] - img.height) // 2
            grid.paste(img, (x, y + y_offset))
        y += row_heights[r]
    return grid


Run the direct decoder baseline over all pages and display the result grid.


In [ ]:
render_pdf_if_needed()
processed = []
total_found = 0

for p in PAGES:
    img_path = PAGE_DIR / f'page-{p:02d}.png'
    if not img_path.exists():
        print(f'Skipping page {p}: image not found')
        continue
    print(f'Processing page {p}...', end=' ')
    img, count = detect_page(img_path)
    print(f'found {count}')
    total_found += count
    processed.append(img)

print(f'\nTotal: {total_found} barcodes across {len(processed)} pages')

if processed:
    grid = create_grid(processed, cols=GRID_COLS, cell_width=CELL_WIDTH)
    print(f'Grid: {grid.size[0]}x{grid.size[1]}')
    display(grid)
else:
    print('No pages processed.')

